# Cross-Validated Segment Model — mRMR + damped class weights

* **Model A — fixed params, 5-fold CV.** One fixed config
  (`C=1, gamma='scale', k_feat=20`) evaluated by stratified 5-fold CV.



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

## Load one patient (seizure files only)

Loads every `*_seizure.mat` file for the patient and keeps the ones that contain
at least one seizure window. Same complex-window handling and 1530-orientation
fix as the honest run.

In [2]:
N_FEATURES = 1530
IMAG_TOL   = 1e-6   # |imag| above this = invalid GC window, dropped

SEIZURE_DIR = Path("Features/Seizure")   # features live in Features/Seizure/ next to the notebook


def _read_feats(h, key="all_feats"):
    """Read features, handling MATLAB complex compound dtype. Returns
    (X real, rows=windows) and a bad-window mask (|imag| too large)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    """Return a seizure_records dict {record_name: (X, y)} for every
    seizure-containing file (records with at least one seizure window)."""
    records = {}
    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        with h5py.File(f, "r") as h:
            X, bad = _read_feats(h, "all_feats")
            yv = h["all_labels"][:].squeeze()
        y = (yv if yv.shape[0] == X.shape[0] else yv[: X.shape[0]]).astype(int)
        if bad.any():
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)
        print("[SEIZURE]", f.stem, X.shape, "positives =", int(y.sum()))
    return {k: v for k, v in records.items() if v[1].sum() > 0}


def pool_windows(seizure_records):
    """Stack every window from a patient's seizure files into one (X, y) pool."""
    X = np.vstack([X for X, _ in seizure_records.values()])
    y = np.hstack([y for _, y in seizure_records.values()]).astype(int)
    return X, y

## Feature selection (FCQ-mRMR), class weighting, and pipeline



In [3]:
def damped_class_weight(y, power=1):
    """Inverse-frequency class weights damped by `power`.
    power=1.0 -> sklearn 'balanced'; power=0.0 -> unweighted; 0.5 -> sqrt.
    Renormalised so the sample-weighted mean weight is 1, so C keeps the
    same effective scale across the ablation. Training labels only."""
    classes, counts = np.unique(y, return_counts=True)
    n = len(y)
    inv = n / (len(classes) * counts)        # 'balanced' inverse-frequency weights
    weights = inv ** power                    # apply the damping exponent
    weights *= n / np.sum(counts * weights)   # renormalise mean sample weight -> 1
    return {int(c): float(w) for c, w in zip(classes, weights)}


# FCQ-mRMR feature selection (aligned to the reference `mrmr` package).
# Redundancy-aware selection: EEG feature banks (band powers x channels x
# derived ratios) are heavily collinear, so pure-relevance top-k (MI) wastes
# the budget on correlated copies of the same signal. mRMR maximises relevance
# while penalising redundancy -> compact, diverse set (better in p >> n).
#   relevance  = ANOVA F-statistic (f_classif)          -- fast, scale-invariant
#   redundancy = mean |Pearson corr| to already-picked features
#   greedy     = argmax  relevance / redundancy  (FCQ quotient variant)
# Deterministic (no RNG) -> also removes a selection-variance source vs MI.
def mrmr_rank(X, y, k, floor=1e-3):
    """Feature indices ordered by greedy FCQ-mRMR (len <= k). Relevance = ANOVA
    F-statistic (f_classif); redundancy = mean |Pearson corr| to already-selected
    features; score = relevance / redundancy (quotient / FCQ form). ALIGNED to the
    reference `mrmr` package (Mazzanti, the FCQ default): candidate pool restricted
    to relevance>0, each pairwise redundancy floored at `floor` (=0.001), and a
    feature whose mean redundancy to the selected set is 1.0 is rejected (score 0).
    Greedy-forward, so order[:m] is the m-feature subset for any m<=k -> compute
    once at max(k) and slice. Matches the package bit-for-bit on well-conditioned
    data; a 1-2 feature residual can appear only in float-tie / exact-duplicate
    edge cases (immaterial)."""
    X = np.asarray(X, float); y = np.asarray(y)
    n, p = X.shape
    with np.errstate(all="ignore"):
        F, _ = f_classif(X, y)
    F = np.nan_to_num(np.asarray(F, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    F = np.maximum(F, 0.0)
    pool = F > 0.0                                      # package: keep only relevance>0
    k = int(min(k, int(pool.sum())))
    if k <= 0:
        return np.array([], dtype=int)
    # z-score (guard zero-variance) so col dot-products == Pearson correlations
    sd = X.std(0); sd = np.where(sd < 1e-12, 1.0, sd)
    Z = (X - X.mean(0)) / sd
    remaining = pool.copy()
    first = int(np.argmax(np.where(pool, F, -np.inf)))  # step 1: most relevant
    selected = [first]; remaining[first] = False
    accum = np.zeros(p)                                 # sum of FLOORED |corr| to picks
    for _ in range(1, k):
        corr = np.abs((Z.T @ Z[:, selected[-1]]) / n)   # |corr| vs last pick
        corr = np.clip(np.nan_to_num(corr, nan=floor), floor, None)   # per-pair floor
        accum += corr
        den = accum / len(selected)                     # mean |corr| to selected
        den = np.where(np.isclose(den, 1.0), np.inf, den)   # reject perfect-redundant
        score = np.where(remaining, F / den, -np.inf)
        nxt = int(np.argmax(score))
        if not np.isfinite(score[nxt]):
            break
        selected.append(nxt); remaining[nxt] = False
    return np.array(selected, dtype=int)


class MRMRSelector(BaseEstimator, TransformerMixin):
    """Top-k FCQ-mRMR selector. Ranking is fit on TRAINING data only, so it is
    leakage-free inside a Pipeline / CV split (drop-in for SelectKBest)."""
    def __init__(self, k=20):
        self.k = k
    def fit(self, X, y=None):
        self.support_idx_ = mrmr_rank(X, y, self.k)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.support_idx_]
# =========================================================================


def make_pipe(cfg, class_weight):
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", MRMRSelector(k=cfg["k_feat"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight=class_weight, cache_size=1024)),
    ])

## Segment metrics


In [ ]:
def _safe_div(a, b):
    return a / b if b else float("nan")


def _segment_confusion(y, sc, thr=0.0):
    """Threshold-`thr` segment confusion metrics (same logic as the honest
    run's segment_metrics): returns acc, spec, sens, prec, f1, tp, fp, fn."""
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def diag_scores(y, sc):
    """ROC-AUC and average precision; NaN if the fold is single-class."""
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)


def segment_row(patient, record, cfg, y, sc):
    """One segment CSV row, column-for-column 
    SEGMENT_PER_RECORD file. `record_length` and `k` are NaN (no intact
    record timeline in a shuffled CV fold)."""
    acc, spec, sens, prec, f1, tp, fp, fn = _segment_confusion(y, sc, 0.0)
    roc, ap = diag_scores(y, sc)
    return {
        "C": cfg["C"], "gamma": cfg["gamma"], "k_feat": cfg["k_feat"],
        "patient": patient, "record": record,
        "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
        "record_length": float("nan"), "threshold": 0.0, "k": float("nan"),
        "roc_auc": roc, "ap": ap,
        "seg_accuracy": acc, "seg_specificity": spec,
        "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
        "tp": tp, "fp": fp, "fn": fn,
    }

## CONFIG

In [ ]:
SEED     = 42
PATIENTS = [f"chb{i:02d}" for i in range(1, 25)]   # chb01 .. chb24
CV_FOLDS = 5
CLASS_WEIGHT_POWER = 0.5

# --- Model A: fixed hyperparameters (nothing tuned) ---
FIXED_CFG = {"C": 1, "gamma": 0.001, "k_feat": 100}

# --- Outputs (segment-level, one row per patient/config/fold) ---
MODEL_A_CSV = "CV_FIXED_segment.csv"


def cv_splits(y, n_splits, seed):
    """Stratified K-fold generator, capping folds at the positive count so a
    patient with few seizure windows still runs. Returns None if <2 positives."""
    n_pos = int(np.sum(y))
    k = min(n_splits, n_pos, len(y) - n_pos)
    if k < 2:
        return None
    return StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

## Model A — fixed params, 5-fold CV

For each patient: pool the seizure-file windows and run stratified 5-fold CV with
the single fixed config. The class weight is recomputed on each training fold.
One segment row per fold.

In [ ]:
def run_model_a(patient, cfg):
    seizure_records = load_patient(patient)
    if not seizure_records:
        print(f"  [{patient}] no seizure records — skipping")
        return []
    X, y = pool_windows(seizure_records)
    skf = cv_splits(y, CV_FOLDS, SEED)
    if skf is None:
        print(f"  [{patient}] too few seizure windows for CV — skipping")
        return []
    print(f"  [{patient}] pool windows={len(y)} positives={int(y.sum())}")

    rows = []
    for i, (tr, te) in enumerate(skf.split(X, y), 1):
        cw = damped_class_weight(y[tr], CLASS_WEIGHT_POWER)
        model = make_pipe(cfg, cw).fit(X[tr], y[tr])
        sc = model.decision_function(X[te])
        row = segment_row(patient, f"fold{i}", cfg, y[te], sc)
        rows.append(row)
        print(f"    fold{i}: AUPRC={row['ap']:.3f} ROC={row['roc_auc']:.3f} "
              f"SENS={row['seg_sensitivity']:.3f} SPEC={row['seg_specificity']:.3f}")
    return rows


rows_a = []
for patient in PATIENTS:
    print(f"\n[A] {patient}")
    try:
        rows_a.extend(run_model_a(patient, FIXED_CFG))
        pd.DataFrame(rows_a).to_csv(MODEL_A_CSV, index=False)
    except Exception as e:
        print(f"  FAILED {patient}: {type(e).__name__}: {e}")

df_a = pd.DataFrame(rows_a)
df_a.to_csv(MODEL_A_CSV, index=False)
print(f"\nWrote {MODEL_A_CSV}  ({len(df_a)} rows)")
df_a